# CS604 Colab — Qian: LoRA r24 + VPT-Deep @ 336

Full-data run of the selected hybrid: LoRA r24 on qkv/proj/fc1/fc2 plus trainable norms, biases, positional embeddings and CLS; **8 independent prompt tokens per block** in all 12 blocks; prototype cosine classifier. All **450,000 training images**, fresh pretrained weights and all-data prototypes. No subset checkpoint or pilot is loaded.

LoRA/extras LR **3e-4**, prompt LR **3e-4** (selected by the pilot), head LR **3e-3**. Batch **128 x accumulation 2**, BF16, seed 42; **144 training minutes**, 5% time warmup + cosine, no epoch ceiling. A100 40GB recommended. Trainable backbone: **4.7391%**, classifier excluded.

Run the cells in order after pushing `qq_branch_v2`. The notebook clones/pulls the branch, downloads data to `/content/data`, validates or builds all-data prototypes, trains, and saves results in Drive. Prediction automatically selects the completed run from this session; set `RUN_NAME` if reopening the notebook. Default prediction is **crop 0.75 with horizontal-flip TTA**, following your previous experiments. Crop/flip settings are at the top.

In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "qq_branch_v2"
METHOD = "lora_vpt_deep_336"
TEAM = "08"
TRAIN_OUT = f"{DRIVE_OUT}/lora_vpt_deep_336_seed42"
# Reuse the existing all-data prototypes at 336 from Drive.
PROTO_PT = f"{DRIVE_OUT}/prototypes_336.pt"
FORCE_REBUILD_PROTOTYPES = False  # leave False to reuse prototypes_336.pt
RESOLVED_CONFIG = "/content/qq_lora_vpt_deep_final_training_config.yaml"
EVAL_CROP_PCT = 0.75
TTA_FLIP = True

## 1. Clone or pull the repo

In [ ]:
import sys
import base64
import os
import subprocess
from getpass import getpass

# Pass credentials through a temporary child-process environment, not the URL.
token = getpass("GitHub PAT (hidden; repo scope). Leave empty for a public repo: ")
git_env = os.environ.copy()
if token:
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraHeader",
                   GIT_CONFIG_VALUE_0=f"Authorization: Basic {auth}")
    del auth
try:
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], env=git_env, check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], env=git_env, check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], env=git_env, check=True)
finally:
    del token, git_env
os.chdir(REPO_DIR)
actual = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
assert actual == BRANCH, f"Wrong branch: {actual}"
print("Branch:", actual)
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

## 2. Install requirements

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

## 4. Download data to `/content/data`

In [ ]:
# Stream subprocess output into the notebook, and stop on failure.
def run_live(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)
    except BaseException:
        if process.poll() is None:
            process.terminate()
        process.wait()
        raise

run_live([sys.executable, "-u", "scripts/download_data.py", "--dest", DATA_DIR])

## 4b. Load all-data prototypes_336.pt

Reuse `CS604_RUNS/prototypes_336.pt` built from the original pretrained backbone at 336 using all 45 images/class. Check backbone, shape, normalization and sample counts; verify resolution/manifest/file hashes when metadata exists. Older files without the metadata sidecar are accepted on the basis of your selection of this 336px file. Build only if absent or explicitly requested. Extraction is outside the 144-minute training timer; subset prototypes must not be used.

In [ ]:
import hashlib
import json
from collections import Counter
from pathlib import Path

import torch
from src.utils import load_config, apply_overrides, save_config

assert torch.cuda.is_available(), "Select a GPU runtime before prototype extraction."

def file_hash(path):
    checksum = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            checksum.update(chunk)
    return checksum.hexdigest()

def resolve_training_config():
    cfg = load_config(f"configs/{METHOD}.yaml")
    apply_overrides(cfg, data_root=DATA_DIR, output_dir=TRAIN_OUT)
    if cfg.get("head_init") == "prototype":
        cfg.prototype_file = PROTO_PT
    save_config(cfg, RESOLVED_CONFIG)
    return cfg

cfg = resolve_training_config()
manifest = Path(DATA_DIR) / "train.txt"
rows = [line.split() for line in manifest.read_text().splitlines() if line.strip()]
assert all(len(row) == 2 for row in rows), "Training manifest must contain path and label."
counts = Counter(int(row[1]) for row in rows)
assert len({row[0] for row in rows}) == len(rows), "Duplicate training image paths."
assert set(counts) == set(range(int(cfg.num_classes))), "Training manifest is missing classes."
assert all(count == 45 for count in counts.values()), "Expected all 45 images per class."
assert len(rows) == 450000, "Expected the complete 450,000-image training set."
print("Training set:", len(rows), "images;", len(counts), "classes; 45 images/class")
val_manifest = Path(DATA_DIR) / "val.txt"
if val_manifest.is_file():
    assert all(len(line.split()) == 1 for line in val_manifest.read_text().splitlines() if line.strip()), \
        "Select the official data root, not a labelled internal split."


prototype_path = Path(PROTO_PT)
metadata_path = prototype_path.with_suffix(".metadata.json")
expected = {"model_name": cfg.model_name, "img_size": int(cfg.img_size),
            "num_classes": int(cfg.num_classes), "training_images": len(rows),
            "train_manifest_sha256": file_hash(manifest)}

def validate_prototypes(config):
    blob = torch.load(prototype_path, map_location="cpu", weights_only=True)
    assert blob.get("model_name") == config.model_name, "Prototype backbone does not match."
    assert tuple(blob["prototypes"].shape) == (int(config.num_classes), 768)
    assert torch.isfinite(blob["prototypes"]).all(), "Nonfinite prototype features."
    assert torch.allclose(blob["prototypes"].float().norm(dim=1),
                          torch.ones(int(config.num_classes)), atol=1e-4), "Prototypes must be normalized."
    assert tuple(blob["samples_per_class"].shape) == (int(config.num_classes),)
    assert torch.all(blob["samples_per_class"] == 45), "Prototypes must use all 45 images/class."
    actual_hash = file_hash(prototype_path)
    metadata = json.loads(metadata_path.read_text()) if metadata_path.is_file() else None
    required = {**expected, "model_name": config.model_name, "img_size": int(config.img_size),
                "num_classes": int(config.num_classes),
                "train_manifest_sha256": file_hash(Path(config.data_root) / "train.txt")}
    if blob.get("img_size") is not None:
        assert int(blob["img_size"]) == int(config.img_size), "Prototype resolution does not match."
    if metadata is not None:
        assert all(metadata.get(key) == value for key, value in required.items()), \
            "Prototype backbone, resolution or full training manifest does not match."
        assert metadata.get("prototype_sha256") == actual_hash, "Prototype file was changed."
    print("Validated prototype contents:", prototype_path, "| metadata present:", metadata is not None)

if cfg.get("head_init") == "prototype":
    if FORCE_REBUILD_PROTOTYPES or not prototype_path.exists():
        # Explicitly disable any per-class sampling inherited from another recipe.
        proto_cfg = load_config(RESOLVED_CONFIG)
        proto_cfg.prototype_per_class = None
        proto_cfg.prompt_mode = "none"
        proto_cfg.prompt_length = 0
        proto_config_path = "/content/qq_lora_vpt_deep_all_data_prototypes_config.yaml"
        save_config(proto_cfg, proto_config_path)
        run_live([sys.executable, "-u", "scripts/build_prototypes.py", "--config", proto_config_path,
                  "--data_root", DATA_DIR, "--output", PROTO_PT, "--batch_size", str(cfg.batch_size)])
        metadata = {**expected, "prototype_sha256": file_hash(prototype_path)}
        metadata_path.write_text(json.dumps(metadata, indent=2))
    validate_prototypes(cfg)
else:
    print("This configuration does not use prototype initialization.")

## Review the selected configuration and parameter budget

Expected: **4,075,008 trainable backbone parameters / 85,986,816 original backbone parameters = 4.7391%**, including **73,728 prompt parameters**. The cosine classifier is excluded. Prompt slots are replaced at each block; they do not accumulate.

In [ ]:
import pandas as pd
from IPython.display import display
from src.model import build_model
from src.peft.lora import LoRALinear
from src.train import adamw_param_groups

chosen = resolve_training_config()
display(pd.DataFrame(list(chosen.to_dict().items()), columns=["parameter", "value"]))

# Check architecture on CPU without downloading pretrained weights.
count_cfg = load_config(RESOLVED_CONFIG)
count_cfg.pretrained = False
count_model = build_model(count_cfg)
PARAMETER_STATS = dict(count_model.adaptation_stats)
PARAMETER_STATS["prompt_parameters"] = count_model.vpt.embeddings.numel()
PARAMETER_STATS["trainable_classifier_parameters"] = sum(
    p.numel() for p in count_model.get_classifier().parameters() if p.requires_grad)
assert count_model.vpt.mode == "deep"
assert tuple(count_model.vpt.embeddings.shape) == (12, 8, 768)
assert sum(isinstance(module, LoRALinear) for module in count_model.modules()) == 48
assert PARAMETER_STATS["original_backbone_parameters"] == 85986816
assert PARAMETER_STATS["adaptation_parameters"] == 4075008
assert PARAMETER_STATS["prompt_parameters"] == 73728
assert PARAMETER_STATS["adaptation_pct"] < 5

groups = adamw_param_groups(count_model, chosen.weight_decay, chosen.lr,
                            head_lr=chosen.head_lr, prompt_lr=chosen.prompt_lr)
assigned = [id(p) for group in groups for p in group["params"]]
assert len(assigned) == len(set(assigned))
assert set(assigned) == {id(p) for p in count_model.parameters() if p.requires_grad}
prompt_group = next(group for group in groups
                    if any(p is count_model.vpt.embeddings for p in group["params"]))
assert prompt_group["lr"] == 3e-4 and prompt_group["weight_decay"] == 0
print("Parameter counts (classifier excluded from cap):", PARAMETER_STATS)
del count_model, groups, prompt_group

## 5. Train on all data for 144 minutes

The budget counts training time; setup, prototype extraction and saving are outside it. Use all 450,000 images with no labelled holdout, fresh pretrained weights, no screening checkpoint and no epoch ceiling. Checkpoints include the trainable prompts and require the original pretrained backbone when loading. The completed run is selected automatically and recorded under `TRAIN_OUT`.

In [ ]:
assert torch.cuda.is_available(), "Select an A100 GPU runtime."
check = resolve_training_config()
print("GPU:", torch.cuda.get_device_name(0))
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Commit:", commit)
print("Resolved training config:", check.to_dict())

assert check.method == "lora" and check.pretrained
assert check.prompt_mode == "deep" and check.prompt_length == 8
assert check.img_size == 336 and check.num_classes == 10000
assert check.lora_rank == 24 and check.lora_alpha == 24 and check.lora_dropout == 0
assert list(check.lora_targets) == ["qkv", "proj", "fc1", "fc2"]
assert list(check.unfreeze_extra) == ["norm", "bias", "pos_embed", "cls_token"]
assert check.cosine_scale == 20.0
assert check.batch_size == 128 and check.grad_accum_steps == 2
assert check.lr == 3e-4 and check.prompt_lr == 3e-4 and check.head_lr == 3e-3
assert check.weight_decay == 0.05 and check.label_smoothing == 0.1
assert check.clip_grad_norm == 1.0 and check.seed == 42 and check.bf16
assert check.epochs is None and check.max_train_minutes == 144
assert check.time_warmup_fraction == 0.05
assert check.head_type == "cosine" and check.head_init == "prototype"
assert file_hash(Path(DATA_DIR) / "train.txt") == expected["train_manifest_sha256"]
validate_prototypes(check)

training_root = Path(TRAIN_OUT)
training_root.mkdir(parents=True, exist_ok=True)
before = set(training_root.glob("lora_*"))
run_live([sys.executable, "-u", "-m", "src.train", "--config", RESOLVED_CONFIG])
created = set(training_root.glob("lora_*")) - before
assert len(created) == 1, f"Expected one new run directory, found {created}"
RUN_DIR = created.pop()
stats = dict(line.split("=", 1) for line in (RUN_DIR / "run_stats.txt").read_text().splitlines())
assert stats["time_limit_reached"] == "True", "Run did not reach the training-time budget"
assert (RUN_DIR / "last.pt").is_file()
(RUN_DIR / "parameter_stats.json").write_text(json.dumps(PARAMETER_STATS, indent=2))
(RUN_DIR / "provenance.json").write_text(json.dumps({
    "commit": commit, "gpu": torch.cuda.get_device_name(0), "torch_version": str(torch.__version__),
    "prototype_file": PROTO_PT, "prototype_sha256": file_hash(PROTO_PT),
    "prototype_metadata_present": metadata_path.is_file(),
    "train_manifest_sha256": expected["train_manifest_sha256"],
}, indent=2))
(training_root / "latest_completed_run.json").write_text(json.dumps({"run_dir": str(RUN_DIR)}, indent=2))
print("Completed run:", RUN_DIR)
print((RUN_DIR / "run_stats.txt").read_text())

## 6. Predict validation and test labels

The completed run from this session is selected automatically. If reopening the notebook, set `RUN_NAME` to its folder name under `TRAIN_OUT`, or leave it empty to load the latest completed-run pointer. Use the saved config and `last.pt`; subset checkpoints are not accepted.

Default: crop **0.75 with horizontal-flip TTA**. Average probabilities from the original and mirrored image. To evaluate another setting, change `EVAL_CROP_PCT` and `TTA_FLIP` above, then rerun this section. Each setting gets its own subfolder, TXT/ZIP files and settings JSON. Test prediction runs only when test data is available.

In [ ]:
from pathlib import Path
import shutil

RUN_NAME = ""  # optional explicit folder name under TRAIN_OUT
if RUN_NAME:
    RUN_DIR = Path(TRAIN_OUT) / RUN_NAME
elif "RUN_DIR" not in globals():
    pointer = Path(TRAIN_OUT) / "latest_completed_run.json"
    assert pointer.is_file(), "Train first, or set RUN_NAME to a completed run."
    RUN_DIR = Path(json.loads(pointer.read_text())["run_dir"])
CKPT = str(RUN_DIR / "last.pt")
assert Path(CKPT).is_file(), f"Checkpoint missing: {CKPT}"

saved_cfg = load_config(RUN_DIR / "config.yaml")
assert saved_cfg.method == "lora" and saved_cfg.img_size == 336
assert saved_cfg.prompt_mode == "deep" and saved_cfg.prompt_length == 8
assert saved_cfg.lora_rank == 24 and saved_cfg.num_classes == 10000
assert saved_cfg.head_type == "cosine" and saved_cfg.prompt_lr == 3e-4
assert saved_cfg.epochs is None and saved_cfg.max_train_minutes == 144
assert saved_cfg.data_root == DATA_DIR, "Select the all-data final run, not a split experiment."
completed_stats = dict(line.split("=", 1) for line in (RUN_DIR / "run_stats.txt").read_text().splitlines())
assert completed_stats["time_limit_reached"] == "True"
PREDICTION_DIR = RUN_DIR / f"crop{EVAL_CROP_PCT:.2f}_flip{int(TTA_FLIP)}"
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
(PREDICTION_DIR / "prediction_settings.json").write_text(json.dumps(
    {"checkpoint": CKPT, "crop_pct": EVAL_CROP_PCT, "tta_flip": TTA_FLIP}, indent=2))

def predict(split):
    command = [sys.executable, "-u", "-m", "src.predict",
               "--config", str(RUN_DIR / "config.yaml"), "--ckpt", CKPT,
               "--split", split, "--team", TEAM, "--data_root", DATA_DIR,
               "--out_dir", str(PREDICTION_DIR), "--crop_pct", str(EVAL_CROP_PCT)]
    if TTA_FLIP:
        command.append("--tta_flip")
    run_live(command)
    name = f"{split}_team_{TEAM}"
    archive = shutil.make_archive(str(PREDICTION_DIR / name), "zip",
                                  root_dir=str(PREDICTION_DIR), base_dir=f"{name}.txt")
    print("Submit:", archive)

predict("val")
if (Path(DATA_DIR) / "test.txt").is_file() and (Path(DATA_DIR) / "test").is_dir():
    predict("test")
else:
    print("Test data not available; skipping test prediction.")

## 7. Loss curves

Plot the losses recorded during training, after training finishes. Save and display the plots in this experiment’s Drive folder. Validation curves require labeled validation.


In [ ]:
from IPython.display import display, Image

FIGURE_DIR = RUN_DIR / "figures"
subprocess.run([
    sys.executable, "scripts/plot_curves.py",
    "--run_dir", str(RUN_DIR), "--out_dir", str(FIGURE_DIR),
], check=True)

for plot in sorted(FIGURE_DIR.glob("*.png")):
    print(plot.name)
    display(Image(filename=str(plot)))

## 8. Inference latency with LoRA merged

Load the trained checkpoint and verify FP32 outputs before/after merging LoRA. **VPT-Deep remains active**. Measure single-view BF16 latency on synthetic images at batch sizes 1 and 64, with 30 warmups and 200 timed iterations. Save `inference_latency_merged.json`. This excludes image loading, preprocessing, transfers and flip/zoom TTA; keep GPU and measurement settings consistent across experiments.

In [ ]:
import json
import torch

from src.utils import load_config, apply_overrides, set_seed
from src.model import build_model
from src.predict import load_checkpoint
from src.peft.lora import merge_lora, LoRALinear
from scripts.benchmark_latency import measure

assert torch.cuda.is_available(), "A GPU runtime is required."
cfg = load_config(RUN_DIR / "config.yaml")
# Partial PEFT checkpoints require the original pretrained backbone.
checkpoint_info = torch.load(CKPT, map_location="cpu", weights_only=False)
partial = bool(checkpoint_info.get("trainable_only", False))
del checkpoint_info
apply_overrides(cfg, pretrained=partial)
set_seed(int(cfg.seed))

device = torch.device("cuda")
model = build_model(cfg).to(device)
load_checkpoint(model, CKPT, device)
model.eval()

sample = torch.randn(1, 3, int(cfg.img_size), int(cfg.img_size), device=device)
with torch.no_grad():
    before_merge = model(sample).float()
    merge_lora(model)
    after_merge = model(sample).float()
merge_error = (before_merge - after_merge).abs().max().item()
torch.testing.assert_close(before_merge, after_merge, atol=2e-4, rtol=2e-4)
assert not any(isinstance(module, LoRALinear) for module in model.modules())
assert model.vpt.mode == "deep" and tuple(model.vpt.embeddings.shape) == (12, 8, 768)
del sample, before_merge, after_merge

warmup = 30
iterations = 200
use_bf16 = bool(cfg.bf16)
results = {
    "checkpoint": CKPT,
    "merged_lora": True,
    "prompt_mode": cfg.prompt_mode,
    "prompt_length": cfg.prompt_length,
    "merge_fp32_max_logit_difference": merge_error,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_capacity_GiB": torch.cuda.get_device_properties(0).total_memory / 1024**3,
    "torch_version": str(torch.__version__),
    "precision": "bf16" if use_bf16 else "fp32",
    "img_size": int(cfg.img_size),
    "warmup_iterations": warmup,
    "timed_iterations": iterations,
    "scope": "Single-view forward pass on synthetic inputs; excludes flip TTA, loading, preprocessing and transfers",
    "measurements": [],
}

try:
    for batch_size in (1, 64):
        batch = torch.randn(
            batch_size, 3, int(cfg.img_size), int(cfg.img_size), device=device,
        )
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_bf16):
            ms_per_image, _ = measure(model, batch, device, warmup, iterations)
        results["measurements"].append({
            "batch_size": batch_size,
            "ms_per_image": ms_per_image,
            "ms_per_batch": ms_per_image * batch_size,
        })
        del batch
finally:
    del model
    torch.cuda.empty_cache()

output = RUN_DIR / "inference_latency_merged.json"
output.write_text(json.dumps(results, indent=2), encoding="utf-8")
print(json.dumps(results, indent=2))
print("Saved:", output)